In [4]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_2604a9a9edd818a6074b6b4df774f0f7'

In [5]:
!kaggle datasets download -d ahluwaliasaksham/car-insurance-fraud-detection-dataset
!unzip car-insurance-fraud-detection-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/ahluwaliasaksham/car-insurance-fraud-detection-dataset
License(s): MIT
car-insurance-fraud-detection-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  car-insurance-fraud-detection-dataset.zip
replace car_insurance_fraud_dataset.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [6]:
import pandas as pd
df = pd.read_csv('car_insurance_fraud_dataset.csv')
df.head()

,policy_id,policy_state,policy_deductible,policy_annual_premium,insured_age,insured_sex,insured_education_level,insured_occupation,insured_hobbies,incident_date,...,incident_state,incident_city,incident_hour_of_the_day,number_of_vehicles_involved,bodily_injuries,witnesses,police_report_available,claim_amount,total_claim_amount,fraud_reported
0,POL100000,GA,400,1430.78,74,OTHER,High School,Manager,reading,2024-06-13,...,MI,Charlesville,6,1,4,0,Yes,8161.36,11677.60,Y
1,POL100001,PA,300,854.49,74,MALE,College,Lawyer,chess,2025-03-23,...,OH,Joshuaberg,0,3,4,5,No,18561.79,18027.81,N
2,POL100002,MI,400,1247.28,28,OTHER,PhD,Doctor,reading,2025-01-26,...,MI,Reynoldsfurt,14,4,4,1,No,10734.61,10375.59,N
3,POL100003,CA,600,622.42,37,MALE,PhD,Teacher,yachting,2024-06-03,...,NC,Josephchester,22,3,3,5,No,13188.92,14204.34,N
4,POL100004,MI,700,1458.17,31,OTHER,PhD,Sales,reading,2024-05-21,...,NY,Caitlinfort,18,4,2,4,No,21864.69,24038.84,N


In [8]:
df.columns.tolist()

['policy_id',
 'policy_state',
 'policy_deductible',
 'policy_annual_premium',
 'insured_age',
 'insured_sex',
 'insured_education_level',
 'insured_occupation',
 'insured_hobbies',
 'incident_date',
 'incident_type',
 'collision_type',
 'incident_severity',
 'authorities_contacted',
 'incident_state',
 'incident_city',
 'incident_hour_of_the_day',
 'number_of_vehicles_involved',
 'bodily_injuries',
 'witnesses',
 'police_report_available',
 'claim_amount',
 'total_claim_amount',
 'fraud_reported']

In [9]:
df.info()
df.isnull().sum()
df['fraud_reported'].value_counts(normalize=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   policy_id                    30000 non-null  object 
 1   policy_state                 30000 non-null  object 
 2   policy_deductible            30000 non-null  int64  
 3   policy_annual_premium        30000 non-null  float64
 4   insured_age                  30000 non-null  int64  
 5   insured_sex                  30000 non-null  object 
 6   insured_education_level      30000 non-null  object 
 7   insured_occupation           30000 non-null  object 
 8   insured_hobbies              30000 non-null  object 
 9   incident_date                30000 non-null  object 
 10  incident_type                30000 non-null  object 
 11  collision_type               30000 non-null  object 
 12  incident_severity            30000 non-null  object 
 13  authorities_cont

,proportion
fraud_reported,
N,0.885333
Y,0.114667


In [10]:
import numpy as np

df['fraud_reported'] = df['fraud_reported'].map({'Y': 1, 'N': 0})
df = df.replace('?', np.nan)
df.isnull().sum()[df.isnull().sum() > 0]

,0
authorities_contacted,7564


In [11]:
df['authorities_contacted'] = df['authorities_contacted'].fillna('Unknown')
df = df.drop(['policy_id'], axis=1)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(categorical_cols)

['policy_state', 'insured_sex', 'insured_education_level', 'insured_occupation', 'insured_hobbies', 'incident_date', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state', 'incident_city', 'police_report_available']


In [12]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df['incident_date'] = pd.to_datetime(df['incident_date'])
df['incident_month'] = df['incident_date'].dt.month
df['incident_day'] = df['incident_date'].dt.day
df = df.drop(['incident_date'], axis=1)

categorical_cols.remove('incident_date')

le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col].astype(str))

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   policy_state                 30000 non-null  int64  
 1   policy_deductible            30000 non-null  int64  
 2   policy_annual_premium        30000 non-null  float64
 3   insured_age                  30000 non-null  int64  
 4   insured_sex                  30000 non-null  int64  
 5   insured_education_level      30000 non-null  int64  
 6   insured_occupation           30000 non-null  int64  
 7   insured_hobbies              30000 non-null  int64  
 8   incident_type                30000 non-null  int64  
 9   collision_type               30000 non-null  int64  
 10  incident_severity            30000 non-null  int64  
 11  authorities_contacted        30000 non-null  int64  
 12  incident_state               30000 non-null  int64  
 13  incident_city   

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

X = df.drop('fraud_reported', axis=1)
y = df['fraud_reported']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

print(y_train.value_counts())
print(y_train_smote.value_counts())

fraud_reported
0    21248
1     2752
Name: count, dtype: int64
fraud_reported
0    21248
1    21248
Name: count, dtype: int64


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import pandas as pd

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes": GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(probability=True, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train_smote)

    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1-Score": f1,
        "ROC-AUC": roc_auc
    })

results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
print(results_df)

                 Model  Accuracy  F1-Score   ROC-AUC
0              XGBoost  0.863000  0.223062  0.678017
1  Logistic Regression  0.677000  0.219178  0.577680
2                  KNN  0.512833  0.203759  0.531799
3                  SVM  0.785333  0.181703  0.579554
4        Decision Tree  0.727667  0.178894  0.523562
5          Naive Bayes  0.720500  0.160240  0.532710
6        Random Forest  0.831167  0.122944  0.593719


In [15]:
import pickle
from xgboost import XGBClassifier

final_model = XGBClassifier(random_state=42)
final_model.fit(X_train_scaled, y_train_smote)

with open('xgboost_fraud_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("تم حفظ الموديل والـ Scaler بنجاح!")

تم حفظ الموديل والـ Scaler بنجاح!
